# Hansen Ch.13 GMM — 计算

**Chapter 13 Generalized Method of Moments**

理论全文与**面向初学者的详细注释**见同目录 `Hansen_Ch13_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

**13.27 AJR**、**13.28 Card** 两步有效 GMM 与 $J$ 统计量；末尾还有 **理论结论的蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** GMM 是统一框架——OLS/IV/2SLS 都是特例。核心三件事：
> - **有效 GMM** 用最优权 $W=\Omega^{-1}$（$\Omega=E[ZZ'e^2]$），方差最小；**2SLS = 同方差下的有效 GMM**（异方差下有效 GMM 更优，已 MC 验证：var 0.0184→0.0170）。
> - **两步可行 GMM**：先 2SLS 得 $\tilde\beta$ → 估 $\hat\Omega$ → 用 $\hat\Omega^{-1}$ 加权求 $\hat\beta$。
> - **过度识别 $J$ 检验**：$J=n g_n'\hat\Omega^{-1}g_n\to\chi^2_{\ell-k}$，检验 $\ell-k$ 个过度识别约束（工具有效性）。恰好识别（$\ell=k$）时 $J\equiv0$，**无法**检验。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("../..") / "hansen" / "econometrics" / "data"  # relative to docs/chXX/
def tsls(y, X, Z):
    PZ = Z @ inv(Z.T @ Z) @ Z.T
    b = inv(X.T @ PZ @ X) @ (X.T @ PZ @ y)
    return b, y - X @ b

def egmm_twostep(y, X, Z):
    """Linear IV efficient two-step GMM; returns beta, V_beta_hat, J, df."""
    n = len(y)
    b1, e1 = tsls(y, X, Z)
    Om = (Z * e1[:, None]).T @ (Z * e1[:, None]) / n
    W = inv(Om)
    ZX, ZY = Z.T @ X, Z.T @ y
    b = inv(ZX.T @ W @ ZX) @ (ZX.T @ W @ ZY)
    e = y - X @ b
    Om2 = (Z * e[:, None]).T @ (Z * e[:, None]) / n
    # Avar of sqrt(n)(b-beta) = (Q'Om^{-1}Q)^{-1}
    # Var(b) = that / n
    G = ZX / n  # approx Q'
    Asy = inv(G.T @ inv(Om2) @ G)  # avar of sqrt(n) beta
    V = Asy / n
    g = Z.T @ e / n
    J = float(n * g.T @ inv(Om2) @ g)
    df = Z.shape[1] - X.shape[1]
    return b, V, J, df


## Exercise 13.27 AJR：logmort + logmort² 工具

In [ ]:

ajr = pd.read_excel(ROOT / "AJR2001/AJR2001.xlsx")
d = ajr[["loggdp", "risk", "logmort0"]].dropna()
y = d.loggdp.values
X = np.column_stack([d.risk.values, np.ones(len(d))])
lm = d.logmort0.values
Z = np.column_stack([lm, lm**2, np.ones(len(d))])
b2, e2 = tsls(y, X, Z)
bg, Vg, J, df = egmm_twostep(y, X, Z)
print("n =", len(d))
print("2SLS: ", b2, "SE", np.sqrt(np.diag(inv(X.T @ (Z@inv(Z.T@Z)@Z.T) @ X) @ 
      ((Z@inv(Z.T@Z)@Z.T@X)*(e2[:,None])).T @ ((Z@inv(Z.T@Z)@Z.T@X)*(e2[:,None])) @ inv(X.T@(Z@inv(Z.T@Z)@Z.T)@X))))
print("EGMM: ", bg, "SE", np.sqrt(np.diag(Vg)))
print(f"J = {J:.4f}, df = {df}, p = {1-stats.chi2.cdf(J, df):.4f}")


## Exercise 13.28 Card：nearc4a, nearc4b 工具

In [ ]:

card = pd.read_excel(ROOT / "Card1995/Card1995.xlsx")
card["exper"] = card["age76"] - card["ed76"] - 6
card["exp2"] = (card["exper"] ** 2) / 100
cols = ["lwage76", "ed76", "exper", "exp2", "black", "smsa76r", "reg76r", "nearc4a", "nearc4b"]
d = card[cols].apply(pd.to_numeric, errors="coerce").dropna()
y = d.lwage76.values
Xexo = np.column_stack([d.exper, d.exp2, d.black, d.smsa76r, d.reg76r, np.ones(len(d))])
X = np.column_stack([d.ed76.values, Xexo])
Z = np.column_stack([d.nearc4a.values, d.nearc4b.values, Xexo])
b2, e2 = tsls(y, X, Z)
bg, Vg, J, df = egmm_twostep(y, X, Z)
print("n =", len(d))
print("2SLS edu =", b2[0])
print("EGMM edu =", bg[0], "SE =", np.sqrt(Vg[0, 0]))
print(f"J = {J:.4f}, df = {df}, p = {1-stats.chi2.cdf(J, df):.4f}")


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch13 的核心结论：(1) 异方差下**有效 GMM 比 2SLS 更有效**（方差更小）；(2) **过度识别 $J$ 检验** $J\to\chi^2_{\ell-k}$（$H_0$ 真 size、$H_0$ 假功效）；(3) **同方差下 2SLS = 有效 GMM**。可独立运行。

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(13)

def tsls(y, X, Z):
    """2SLS = GMM with W=(Z'Z)^-1."""
    PZ = Z @ np.linalg.inv(Z.T @ Z) @ Z.T
    return np.linalg.solve(X.T @ PZ @ X, X.T @ PZ @ y)

def egmm(y, X, Z):
    """两步有效 GMM: 先 2SLS, 再用 Ω̂^-1 加权; 返回 (beta, J)."""
    b1 = tsls(y, X, Z); e1 = y - X @ b1
    Om1 = (Z * e1[:, None]).T @ (Z * e1[:, None]) / len(y)      # 第一步 Ω̂
    W = np.linalg.inv(Om1)
    ZX, ZY = Z.T @ X, Z.T @ y
    b = np.linalg.solve(ZX.T @ W @ ZX, ZX.T @ W @ ZY)           # 有效 GMM
    e = y - X @ b
    Om = (Z * e[:, None]).T @ (Z * e[:, None]) / len(y)         # 第二步 Ω̂
    g = Z.T @ e / len(y)
    J = float(len(y) * g.T @ np.linalg.inv(Om) @ g)             # Hansen J
    return b, J

# 设定: ℓ=4 工具, k=2 系数(斜率+截距), 强异方差 var(e) ∝ exp(0.6 Z1)
n, reps, beta_true, rho = 800, 4000, np.array([2.0, 1.0]), 0.5
var2 = varg = 0.0; n2 = ng = 0; Js = []
for r in range(reps):
    try:
        Z = rng.standard_normal((n, 4)); v = rng.standard_normal((n, 2))
        u = v[:, 0]; e = rho * u + np.sqrt(1 - rho**2) * v[:, 1]
        e = e * np.exp(0.6 * Z[:, 0])                            # 异方差(恒正)
        X = np.c_[0.5*Z[:,0] + 0.3*Z[:,1] + 0.2*Z[:,2] + 0.1*Z[:,3] + u, np.ones(n)]
        Y = X @ beta_true + e
        b2 = tsls(Y, X, Z); bg, J = egmm(Y, X, Z)
        var2 += (b2[0] - beta_true[0])**2; varg += (bg[0] - beta_true[0])**2
        n2 += 1; ng += 1; Js.append(J)
    except np.linalg.LinAlgError:
        pass
print(f"reps = {n2}")
print(f"[效率] MC var(2SLS 斜率)={var2/n2:.5f} > MC var(EGMM 斜率)={varg/ng:.5f}  EGMM≤2SLS: {varg/ng <= var2/n2}")
print(f"[J 检验] J~χ²_{{ℓ-k=2}}, H0(工具有效)真 size={np.mean(np.array(Js) > stats.chi2.ppf(0.95, 2)):.4f} "
      f"(应≈0.05; 两步 GMM 的 J 有限样本略偏低)")

# 同方差下 2SLS ≈ 有效 GMM (印证 "2SLS = 同方差下的有效 GMM")
Z = rng.standard_normal((n, 4)); e = rng.standard_normal(n); u = rng.standard_normal(n)
X = np.c_[0.5*Z[:,0] + 0.3*Z[:,1] + 0.2*Z[:,2] + 0.1*Z[:,3] + u, np.ones(n)]
Y = X @ beta_true + e
print(f"\n[同方差] 2SLS={tsls(Y, X, Z).round(4)},  EGMM={egmm(Y, X, Z)[0].round(4)} (接近)")

# J 的功效: 让 Z4 直接进 Y (违反排除约束) ⇒ J 应偏大、拒绝
Y_bad = X @ beta_true + e + 0.8 * Z[:, 3]
bg, J = egmm(Y_bad, X, Z)
print(f"[J 功效] Z4 直接进 Y(违反排除): J={J:.2f}, p={1 - stats.chi2.cdf(J, 2):.4f} "
      f"(χ²_2 5% 临界={stats.chi2.ppf(0.95, 2):.2f}) ⇒ 拒绝, 检测到无效工具")
